#BRONZE INGESTION — E-Commerce Lakehouse

In [0]:
from pyspark.sql.functions import *

In [0]:
df = spark.read.format("csv")\
        .option("header", True)\
            .option("inferSchema", True)\
                .load("/Volumes/workspace/default/e-commerce/olist_orders_dataset.csv")

In [0]:
df.limit(5).display()

In [0]:
df.printSchema()

In [0]:
df = df.withColumn("_load_timestamp", current_timestamp())\
          .withColumn("_source_file", col("_metadata.file_path"))

df.limit(5).display()
    

## Configs

In [0]:
base_path = "/Volumes/workspace/default/e-commerce"

file_to_table = {
    "olist_orders_dataset.csv":        "orders",
    "olist_order_items_dataset.csv":   "order_items",
    "olist_customers_dataset.csv":     "customers",
    "olist_products_dataset.csv":      "products",
    "olist_order_payments_dataset.csv":"payments",
    # "olist_order_reviews_dataset.csv": "reviews",
}

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

for file_name, table_name in file_to_table.items():
    source = f"{base_path}/{file_name}"

    # 1. Read raw — schema-on-read, capture as-is
    df = (
        spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(source)
    )

    # 2. Add lineage IMMEDIATELY, while file context is still available.
    #    _metadata.file_path works on Unity Catalog; input_file_name() is
    #    use the built-in _metadata column.
    df = (
        df.withColumn("_load_timestamp", current_timestamp())
          .withColumn("_source_file", col("_metadata.file_path"))
    )

    # 3. Write to Delta. overwrite => deterministic re-runs on a static dataset.
    df = (
        spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .option("multiLine", True)
        .option("quote", '"')
        .option("escape", '"')
        .load(source)
    )

    print(f"bronze.{table_name}  ←  {file_name}  ({df.count():,} rows)")

print("\nBronze ingestion complete.")

In [0]:
display(spark.table("bronze.orders").select("order_id", "_load_timestamp", "_source_file").limit(5))